In [845]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [846]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, OPTICS, AffinityPropagation 
from sklearn.cluster import estimate_bandwidth # for MeanShift clustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder,OneHotEncoder, LabelEncoder # preprocessing
from sklearn.metrics import silhouette_score, adjusted_rand_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # transform columns
import time # to calculate execution time
from scipy.optimize import linear_sum_assignment # match cluster labels to true class labels optimally
from joblib import Parallel, delayed
import itertools # to create combinations of hyperparameters
from sklearn.base import BaseEstimator, TransformerMixin, ClusterMixin # for Affinity Propagation and MeanShift clustering
from sklearn.metrics import pairwise_distances # for Affinity Propagation




### Openml
In Python, OpenML is mainly used to discover, download, and share ML datasets, tasks, and results—super handy for experiments, benchmarking, and learning ML properly.

In [847]:
%pip install openml



In [848]:
import openml

### Download the dataset from openl using dataset id

In [849]:
def download_dataset(dataset_id):
    dataset = openml.datasets.get_dataset(dataset_id)
    X,y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute)
    return X,y, categorical_indicator, attribute_names


### Prepare data
Define numerical and categorical columns based on the categorical_indicator

In [850]:
# Define columns types
def define_column_types(X):
    cat_columns = X.columns[np.array(categorical_indicator)==True]
    num_columns = X.columns[np.array(categorical_indicator)==False]
    return list(num_columns), list(cat_columns)

# Pre-processing
Numeric variables --> scale
Ordinal variables -->  Ordinal encoding
Nominal categorical variables -->  Onehot encoding
Binary(already 0 and 1) -->  onehot encoding 


# Define parameters

In [851]:
def define_parameters(algorithm):
    if algorithm == "KMeans":
        param_grid = {
        "n_clusters": [2, 3, 4, 5, 6],
        "init": ["k-means++", "random"],
        "n_init": [10, 20],
        "max_iter": [300, 500]
        }

    if algorithm == "AgglomerativeClustering":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["single","average", "complete"],
        "metric": ["euclidean", "manhattan"] 
        }

    if algorithm == "Ward":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["ward"], # only ward can be used
        "metric": ["euclidean"] # only euclidean can be used
        }
    
    if algorithm == "DBSCAN":
        param_grid = {
            "eps": [0.1,0.5, 0.7,0.8],
            "min_samples": [10, 20, 30],
            "metric": ["euclidean"],
            "algorithm":["auto", "ball_tree", "kd_tree", "brute"]
        }

    if algorithm == "OPTICS":
        param_grid={
            "min_samples": [10,20,30],
            "max_eps": [0.5],
            "xi": [0.05, 0.1],
            "min_cluster_size": [10, 20],
            "metric": ["euclidean"],
            "algorithm":["auto", "ball_tree", "kd_tree", "brute"]
        }
    
    if algorithm == "GaussianMixture":
        param_grid = {
            "n_components": [2,3,4], # clusters
            "covariance_type": ["full", "tied"],
            "init_params" : ['kmeans', 'random']
        }

    if algorithm == "AffinityPropagation":
        param_grid = {
            "damping": [0.5, 0.7], # Damping factor to stabilize updates (0.5–1.0). Prevents oscillations.
            "max_iter": [300, 500], # Maximum number of iterations
            "convergence_iter": [10,20], # Number of iterations with no change in cluster assignments to declare convergence
             "affinity": ['euclidean','precomputed'] # Number of iterations with no change in cluster assignments to declare convergence
        }

    if algorithm == "MeanShift":
        param_grid = {
            "quantile": [0.1, 0.2, 0.3],
            "bin_seeding": [True, False],
            "n_samples": [30,40] 
        }
    
        
    param_names = list(param_grid.keys())
    param_combinations =list(itertools.product(    
        *(param_grid[param_name] for param_name in param_names))
        )
    return  param_grid, param_combinations, param_names


### Define similarity matrix for Affinity Propagation

In [852]:
class SimilarityTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, metric='euclidean'):
        self.metric = metric  # The distance metric to use for computing similarities
    
    def fit(self, X, y=None):
        # Nothing to learn here; just return self
        return self
    
    def transform(self, X):
        # Compute negative pairwise distances as a similarity matrix
        similarity_matrix = -pairwise_distances(X, metric=self.metric)
        return similarity_matrix

### create a custom andwidth estimator for MeanShift clustering

In [853]:
class MeanShiftAutoBW(BaseEstimator, ClusterMixin):
    """
    Mean-Shift clustering wrapper for pipelines.
    Allows bandwidth estimation from a quantile.
    """
    def __init__(self, quantile=0.2, n_samples=400, bin_seeding=True):
        self.quantile = quantile
        self.n_samples = n_samples
        self.bin_seeding = bin_seeding

    def fit(self, X, y=None):
        # Estimate bandwidth from data
        self.bandwidth_ = estimate_bandwidth(
            X,
            quantile=self.quantile,
            n_samples=self.n_samples
        )

        # Fit Mean-Shift with the computed bandwidth
        self.model_ = MeanShift(
            bandwidth=self.bandwidth_,
            bin_seeding=self.bin_seeding
        )
        self.model_.fit(X)

        # Store labels
        self.labels_ = self.model_.labels_
        return self

    def predict(self, X):
        """
        Mean-Shift does not support true prediction.
        We return labels for the fitted data only.
        """
        return self.labels_

# Define model

In [854]:
def evaluate_performance(params):
    start = time.time()

    keys = param_names
    params_dict = dict(zip(keys, params))
    #print(params_dict)

    if algorithm in ["KMeans", "GaussianMixture"]:
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict, random_state=42))
            ])
    if algorithm == "AffinityPropagation":
         pipe = Pipeline([
            ("preprocess" , preprocessor),
            ("similarity", SimilarityTransformer(metric='euclidean')),
            (algorithm, algorithms[algorithm](**params_dict, random_state=42))
            ])
    else:
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict))
            ])
    
    
    pipe.fit(X)
    # Access the last step (KMeans model)
    last_step = pipe.steps[-1][1]
    y_pred = last_step.predict(X) 
    
    #y_pred = pipe.fit_predict(X)

    # Filter the predicted noise cluster(-1) from labels and features
    # mask = y_predicted !=1 # create a mask to remove clusters = -1 which is the noise detected by DBSCAN algorithm
    # y_pred =y_predicted[mask] # filter the noise from labels
    # X_clustered = X[mask] # filter the noise from features
   
    silhouette = silhouette_score(X, y_pred)

    # # silhouette_score
    # n_clusters = np.unique(y_pred)
    # if n_clusters >= 2:
    #     silhouette = silhouette_score(X, y_pred)
    # else:
    #     silhouette = None
    
    # adjusted_rand_score
    ari = adjusted_rand_score(y_enc, y_pred) # no need to do label alignment

    # f1_score
    # Align cluster labels to true labels using Hungarian algorithm
    def align_cluster_labels(y_enc, y_pred):
        cm = confusion_matrix(y_enc, y_pred)  
        row_ind, col_ind = linear_sum_assignment(-cm) # # Maximize diagonal → minimize negative
        mapping = {col: row for row, col in zip(row_ind, col_ind)} 
        return np.array([mapping[label] for label in y_pred])

    y_pred_aligned = align_cluster_labels(y_enc, y_pred)
    f1 = f1_score(y_enc, y_pred_aligned, average="macro") # Computes F1 per class, takes an unweighted mean, treats all clusters/classes equally
    
   
    execution_time = float(time.time() - start)
    #params_dict["inertia"] = pipe.named_steps[algorithm].inertia_
    params_dict["silhouette_score"] = silhouette
    params_dict["adjusted_rand_score"] = ari
    params_dict["f1_score"] = f1
    params_dict["execution_time"] = execution_time
    

    return params_dict



### Joblib 
joblib is a Python library mainly used in ML for saving models, fast loading, and parallel processing

n_jobs answers “how many things can run in parallel? n_jobs does NOT say threads or processes.
Threading is how parallelism is done. Threading is a backend choice in joblib.
backend="threading"   # threads
backend="loky"        # processes (default)

### Create lists of openml datasets and datasets to be analyzed

In [855]:
dataset_names  = ["iris", "wine"] # datasets to be analyzed

datasets = openml.datasets.list_datasets(output_format="dataframe") # openml datasets

In [856]:
algorithms = {
    "KMeans": KMeans, 
    "AgglomerativeClustering": AgglomerativeClustering,
    "Ward": AgglomerativeClustering,
    "DBSCAN": DBSCAN,
    "OPTICS": OPTICS,
    "GaussianMixture": GaussianMixture,
    "AffinityPropagation": AffinityPropagation,
    "Meanshift": MeanShiftAutoBW
}

results_all = pd.DataFrame()

for algorithm in algorithms:
    for dataset_name in dataset_names:
        dataset_id = int(datasets.loc[datasets["name"] == dataset_name, "did"].values[0])
        # Download dataset
        X,y, categorical_indicator, attribute_names = download_dataset(dataset_id)

        # define trypes of columns in X
        num_columns, cat_columns = define_column_types(X) 

        # preprocess y
        label_enc = LabelEncoder()
        y_enc = label_enc.fit_transform(y)

        # define preprocessor for X
        preprocessor = ColumnTransformer(
        transformers = [
            ("num", MinMaxScaler(), num_columns),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
        ]
        )
        
        # define parameter grid
        param_grid, param_combinations, param_names = define_parameters(algorithm)
    
        # evaluate performance of each algorithm
        results = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluate_performance)(params) for params in param_combinations
        )
        results_df = pd.DataFrame(results)
        results_df.insert(0,"dataset", dataset_name)
        results_df.insert(1,"algorithm", algorithm)
        results_all = pd.concat([results_all, results_df], ignore_index=True)
        print(results_all[results_all["dataset"]=="wine"])
        
results_all.to_csv("results.csv")

            
    
   

 


    


    



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  11 out of  38 | elapsed:   10.5s remaining:   25.8s
[Parallel(n_jobs=-1)]: Done  15 out of  38 | elapsed:   10.5s remaining:   16.1s
[Parallel(n_jobs=-1)]: Done  19 out of  38 | elapsed:   10.5s remaining:   10.5s
[Parallel(n_jobs=-1)]: Done  23 out of  38 | elapsed:   10.5s remaining:    6.8s
[Parallel(n_jobs=-1)]: Done  27 out of  38 | elapsed:   10.5s remaining:    4.2s
[Parallel(n_jobs=-1)]: Done  31 out of  38 | elapsed:   10.5s remaining:    2.3s
[Parallel(n_jobs=-1)]: Done  35 out of  38 | elapsed:   10.5s remaining:    0.8s


ValueError: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)